# Fine-tune wav2vec2-base cho phát âm tiếng Anh của người Việt

Notebook chạy **trên Google Colab có GPU**. Máy local chỉ chuẩn bị dữ liệu và code.

1. Upload `data/generated/colab/mdd-colab.zip` và file `.sha256` đi kèm vào `MyDrive/MDD/`.
2. Upload notebook này lên Colab. Chọn **Runtime → Change runtime type → GPU**.
3. Chạy phần chuẩn bị, sau đó train **A** trước. Đổi `STAGE` thành **B**, rồi **C** cho từng `FOLD` và chạy lại các cell chọn thí nghiệm/huấn luyện.
4. Chỉ bật đánh giá test sau khi đã khóa cấu hình bằng validation.

Chỉ dùng checkpoint **facebook/wav2vec2-base**, 3.599 câu có nhãn thủ công và split theo người nói. Không cần key LLM/TTS. Đây là protocol riêng của project, không phải tái lập nguyên trạng điểm benchmark của paper.

Checkpoint, optimizer/RNG để resume, log và dự đoán lưu trên Drive. Audio được giải nén vào ổ VM Colab để tránh đọc hàng nghìn file nhỏ trực tiếp từ Drive.

In [ ]:
from pathlib import Path

DRIVE_FOLDER = "MDD"  # thư mục trực tiếp trong MyDrive
BUNDLE_FILENAME = "mdd-colab.zip"
STAGE = "A"           # A: nguồn; B: thích ứng tiếng Việt; C: đối chứng; smoke: kiểm tra GPU
SEED = 42             # dùng cùng seed cho A/B/C; có thể chạy lại 123, 2026 ở run mới
FOLD = 1              # 1, 2, 3, 4; A chỉ train một lần và dùng chung
RESUME = False        # True để tiếp tục chính run đang dở, không tạo thí nghiệm mới
MAX_STEPS = None       # None dùng ngân sách config; không đổi giữa B/C cùng thí nghiệm
FINAL_EVALUATE_TEST = False  # chỉ True sau khi đã khóa mọi lựa chọn bằng validation

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
RUNS_ROOT = DRIVE_ROOT / "runs"
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
print("Lưu kết quả tại:", RUNS_ROOT)

## Kiểm tra gói và giải nén

Cell kiểm tra SHA-256 toàn ZIP trước khi giải nén, rồi đối chiếu từng file với manifest trong gói. Gói chỉ gồm dữ liệu annotated, model BASE và code cần để train; giữ nguyên đường dẫn và nhãn gốc.

In [ ]:
import hashlib
import shutil
import stat
import subprocess
import sys
import zipfile


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


bundle_drive = DRIVE_ROOT / BUNDLE_FILENAME
checksum_file = bundle_drive.with_name(bundle_drive.name + ".sha256")
if not bundle_drive.is_file() or not checksum_file.is_file():
    raise FileNotFoundError(f"Upload ZIP và .sha256 vào {DRIVE_ROOT} trước khi tiếp tục")
expected_digest = checksum_file.read_text().split()[0]
if len(expected_digest) != 64:
    raise ValueError("File .sha256 không đúng định dạng")
local_bundle = Path("/content") / (expected_digest + ".zip")
if not local_bundle.exists() or file_sha256(local_bundle) != expected_digest:
    shutil.copyfile(bundle_drive, local_bundle)
if file_sha256(local_bundle) != expected_digest:
    raise ValueError("ZIP không khớp checksum; upload lại cả ZIP và .sha256")
workspace = Path("/content/mdd-workspaces") / expected_digest[:16]
ROOT = workspace / "MDD"
if not (ROOT / "bundle_manifest.json").is_file():
    workspace.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local_bundle) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError("ZIP có tên file trùng")
        for info in archive.infolist():
            relative = Path(info.filename)
            if (relative.is_absolute() or ".." in relative.parts
                    or not relative.parts or relative.parts[0] != "MDD"
                    or stat.S_ISLNK(info.external_attr >> 16)):
                raise ValueError(f"Đường dẫn ZIP không hợp lệ: {info.filename}")
        archive.extractall(workspace)
subprocess.run([sys.executable, str(ROOT / "scripts/export_colab_bundle.py"),
                "--verify-extracted", str(ROOT)], check=True, cwd=ROOT)
print("Code và audio nằm trên VM:", ROOT)

## Dependency và kiểm tra GPU

Giữ PyTorch CUDA có sẵn của Colab. **Không cài `requirements/cpu-torch.txt` trên Colab.** Các package còn lại được khóa phiên bản. Chạy các lệnh train trong subprocess để dùng đúng package vừa cài và tránh module cũ trong kernel notebook.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r",
                str(ROOT / "requirements/colab.txt")], check=True)
# Colab có nhiều package cài sẵn không dùng trong project; lưu pip check để chẩn đoán.
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], text=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(pip_check.stdout or "pip check: OK")
GPU_CHECK = r'''import json, torch
from transformers import Wav2Vec2ForCTC
parts = torch.__version__.split("+")[0].split(".")
assert tuple(map(int, parts[:2])) >= (2, 6), "Cần torch >= 2.6 để nạp checkpoint .bin"
assert torch.cuda.is_available(), "Chưa có GPU: chọn Runtime > Change runtime type > GPU"
print(json.dumps({"torch":torch.__version__, "cuda":torch.version.cuda,
                  "gpu":torch.cuda.get_device_name(0),
                  "vram_gib":round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2)}, indent=2))
'''
subprocess.run([sys.executable, "-c", GPU_CHECK], check=True, cwd=ROOT)

In [ ]:
import json
import os
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SETUP_DIR = DRIVE_ROOT / "setup" / stamp
SETUP_DIR.mkdir(parents=True, exist_ok=True)
(SETUP_DIR / "bundle.sha256").write_text(expected_digest + "\n")
(SETUP_DIR / "pip-check.txt").write_text(pip_check.stdout)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True,
                        text=True, stdout=subprocess.PIPE)
(SETUP_DIR / "requirements-colab.lock.txt").write_text(freeze.stdout)
TRAIN_ENV = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                 TOKENIZERS_PARALLELISM="false", OMP_NUM_THREADS="2", MKL_NUM_THREADS="2")


def run_command(arguments, log_path=None):
    command = [sys.executable, "-u", *map(str, arguments)]
    print("Chạy:", " ".join(command))
    if log_path is None:
        subprocess.run(command, check=True, cwd=ROOT, env=TRAIN_ENV)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=ROOT, env=TRAIN_ENV, text=True,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, command)
        except BaseException:
            process.terminate()
            process.wait()
            raise


run_command(["scripts/check_training_environment.py", "--output", SETUP_DIR / "environment.json"])
run_command(["scripts/download_models.py", "--verify-only"])
run_command(["scripts/prepare_l2arctic.py", "--verify-audio"])
run_command(["-m", "unittest", "discover", "-s", "tests", "-v"])

## Chọn thí nghiệm và train

- **A** học phoneme trên `source_train` (2.249 câu), chọn checkpoint bằng `source_dev` (750 câu).
- **B** khởi tạo từ checkpoint tốt nhất của A, học tiếp trên 300 câu Việt train của fold và replay dữ liệu nguồn.
- **C** khởi tạo từ cùng A, học tiếp chỉ dữ liệu nguồn với cùng ngân sách update; chọn bằng cùng Vietnamese validation với B.
- **smoke** chỉ là kiểm tra kỹ thuật ngắn trên GPU, không đưa vào bảng kết quả nghiên cứu.

Chạy A một lần. Sau đó chạy B/C cho fold 1–4. `RESUME=True` tiếp tục run đang dở từ checkpoint cuối, gồm optimizer và trạng thái ngẫu nhiên; không đổi config của run. Nếu thay cấu hình thí nghiệm, đặt run ID mới trong cell dưới và giữ nguyên run cũ làm bằng chứng.

Chạy cell train sẽ bắt đầu cập nhật trọng số trên **GPU Colab**. Model tải nguyên bản có head phoneme mới, nên chưa có độ chính xác MDD trước giai đoạn A.

In [ ]:
if STAGE not in {"A", "B", "C", "smoke"} or FOLD not in {1, 2, 3, 4}:
    raise ValueError("STAGE phải là A/B/C/smoke; FOLD từ 1 đến 4")
config_names = {"A":"source.json", "B":"adapt_vi.json", "C":"control.json", "smoke":"smoke.json"}
CONFIG = ROOT / "configs" / config_names[STAGE]
RUN_ID = f"{STAGE}-seed{SEED}" if STAGE in {"A", "smoke"} else f"{STAGE}-fold{FOLD}-seed{SEED}"
RUN_PATH = RUNS_ROOT / RUN_ID
train_arguments = ["scripts/train_mdd.py", "--config", CONFIG,
                   "--run-id", RUN_ID, "--runs-root", RUNS_ROOT, "--device", "cuda", "--fold", FOLD, "--seed", SEED]
if MAX_STEPS is not None:
    train_arguments += ["--max-steps", int(MAX_STEPS)]
if RESUME:
    resume_checkpoint = RUN_PATH / "checkpoint"
    if not (resume_checkpoint / "training_state.pt").is_file():
        previous_checkpoint = RUN_PATH / "checkpoint.previous"
        if not (previous_checkpoint / "training_state.pt").is_file():
            raise FileNotFoundError(f"Chưa có checkpoint hoàn chỉnh để resume: {RUN_PATH}")
        resume_checkpoint = previous_checkpoint
        print("Dùng bản checkpoint trước do lần lưu cuối bị gián đoạn:", resume_checkpoint)
    train_arguments += ["--resume", resume_checkpoint]
elif STAGE in {"B", "C"}:
    source_checkpoint = RUNS_ROOT / f"A-seed{SEED}" / "best_checkpoint"
    if not source_checkpoint.is_dir():
        raise FileNotFoundError("Cần train A xong trước; chưa có A-seed42/best_checkpoint")
    train_arguments += ["--init-checkpoint", source_checkpoint]
print("Run:", RUN_ID, "| fold:", FOLD, "| resume:", RESUME)
print(json.dumps(json.loads(CONFIG.read_text()), ensure_ascii=False, indent=2))

In [ ]:
run_command(train_arguments, RUNS_ROOT / "_console" / (RUN_ID + ".log"))
print("Hoàn tất run:", RUN_PATH)
print("Checkpoint cuối:", RUN_PATH / "checkpoint")
print("Checkpoint chọn bằng validation:", RUN_PATH / "best_checkpoint")

## Đánh giá validation

Dùng validation để kiểm tra pipeline và chọn tham số. Kết quả này không phải test cuối. Khi chạy A, cell dưới đánh giá trên source validation; với B/C dùng Vietnamese validation của fold.

In [ ]:
validation_split = "source_validation" if STAGE in {"A", "smoke"} else "validation"
validation_arguments = ["scripts/evaluate_mdd.py", "--checkpoint", RUN_PATH / "best_checkpoint",
                        "--split", validation_split, "--fold", FOLD,
                        "--output-dir", RUN_PATH / "evaluation" / f"fold{FOLD}-{validation_split}", "--device", "cuda"]
if STAGE == "smoke":
    validation_arguments += ["--max-samples", 2]
run_command(validation_arguments)


## Test cuối trên cùng ID cho A/B/C

Giữ `FINAL_EVALUATE_TEST=False` trong quá trình phát triển. Khi đã khóa tham số và train đủ A/B/C, bật True và chạy cell này một lần cho từng fold. Cả ba model được chấm trên cùng người Việt test; mỗi dự đoán và bộ đếm được lưu để kiểm tra lại. Không dùng test để chọn checkpoint hoặc chỉnh prompt/hyperparameter.

Điểm F1/precision/recall càng cao càng tốt; PER/FAR/FRR/DER càng thấp càng tốt. Rates trong JSON là tỷ lệ 0–1, PER có thể lớn hơn 1. Không coi loss forward hoặc kết quả smoke là bằng chứng fine-tune cải thiện.

In [ ]:
if not FINAL_EVALUATE_TEST:
    print("Bỏ qua test: chỉ bật FINAL_EVALUATE_TEST sau khi khóa cấu hình bằng validation.")
else:
    final_runs = {"A":f"A-seed{SEED}", "B":f"B-fold{FOLD}-seed{SEED}", "C":f"C-fold{FOLD}-seed{SEED}"}
    for label, run_id in final_runs.items():
        checkpoint = RUNS_ROOT / run_id / "best_checkpoint"
        if not checkpoint.is_dir():
            raise FileNotFoundError(f"Chưa có checkpoint {label}: {checkpoint}")
    for label, run_id in final_runs.items():
        destination = RUNS_ROOT / run_id / "evaluation" / f"fold{FOLD}-test"
        run_command(["scripts/evaluate_mdd.py", "--checkpoint", RUNS_ROOT / run_id / "best_checkpoint",
                     "--split", "test", "--fold", FOLD, "--output-dir", destination, "--device", "cuda"],
                    destination.parent / f"fold{FOLD}-test-console.log")
    print("Đã lưu dự đoán và metrics A/B/C cho fold", FOLD, "tại", RUNS_ROOT)
    comparison_dir = RUNS_ROOT / "comparisons" / f"fold{FOLD}-seed{SEED}"
    run_command(["scripts/compare_mdd_runs.py",
                 "--a", RUNS_ROOT / final_runs["A"] / "evaluation" / f"fold{FOLD}-test",
                 "--b", RUNS_ROOT / final_runs["B"] / "evaluation" / f"fold{FOLD}-test",
                 "--c", RUNS_ROOT / final_runs["C"] / "evaluation" / f"fold{FOLD}-test",
                 "--output", comparison_dir])
    print("Bảng so sánh đã kiểm tra cùng ID/audio/gold/budget:", comparison_dir / "comparison.csv")


## Giữ bằng chứng và tiếp tục khi Colab ngắt

Tải về nguyên thư mục `MyDrive/MDD/runs/`, gồm checkpoint tốt nhất/cuối, config, manifest/hash, môi trường, training log, metrics và dự đoán từng câu. Giữ cả B tốt hơn hoặc kém hơn C.

Nếu Colab ngắt: kết nối GPU mới, chạy lại các cell chuẩn bị, chọn đúng STAGE/FOLD/SEED, đặt `RESUME=True`, chạy lại cell chọn run và train. Dữ liệu VM được giải nén lại khi cần; checkpoint nằm trên Drive. Resume từ checkpoint cuối đã ghi thành công, các update sau lần lưu cuối có thể phải tính lại.

Colab không bảo đảm loại GPU và thời lượng phiên. Tham khảo [Colab FAQ](https://research.google.com/colaboratory/faq.html). Nếu hết VRAM, dùng batch vật lý 1 và accumulation phù hợp trong một config/run mới; áp dụng cùng cấu hình cho B/C, không cắt audio trong khi giữ target đầy đủ.

Notebook đã được kiểm tra cấu trúc/cú pháp ở local; việc train và đánh giá trên GPU chỉ được xác nhận khi bạn chạy notebook trên Colab. Không có điểm MDD thật được điền sẵn.